# 04 — Fine-Tuning DistilBERT on IMDb
## ⚠️ Run this notebook on Google Colab (Runtime → Change runtime type → T4 GPU)

This notebook fine-tunes `distilbert-base-uncased` on the full IMDb training set.  
Expected time: ~20 min/epoch on T4 GPU. We train for 3 epochs with early stopping.


In [ ]:
# Run this cell first on Colab
import subprocess
subprocess.run(["pip", "install", "-q", "transformers", "datasets", "accelerate"])


In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
from sklearn.metrics import accuracy_score, f1_score

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


## Load & Tokenize Dataset

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

raw = load_dataset("imdb")

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)

tokenized = raw.map(tokenize, batched=True, remove_columns=["text"])
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch")

train_ds = tokenized["train"]
test_ds  = tokenized["test"]

print(f"Train: {len(train_ds):,}  |  Test: {len(test_ds):,}")


## Define Metrics

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": round(accuracy_score(labels, preds), 4),
        "f1":       round(f1_score(labels, preds, average="weighted"), 4),
    }


## Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="distilbert-imdb",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    logging_steps=200,
    report_to="none",
)


## Initialize Model & Trainer

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)


## Train

In [ ]:
trainer.train()


## Final Evaluation

In [ ]:
results = trainer.evaluate()
print(results)


## Save Model (optional — download from Colab)

In [ ]:
trainer.save_model("distilbert-imdb-final")
tokenizer.save_pretrained("distilbert-imdb-final")
print("Model saved to distilbert-imdb-final/")

# Download from Colab using:
# from google.colab import files
# import shutil
# shutil.make_archive("distilbert-imdb-final", "zip", "distilbert-imdb-final")
# files.download("distilbert-imdb-final.zip")
